In [ ]:
import MDAnalysis as mda
import MDAnalysis.analysis.rms as rms
import MDAnalysis.analysis.rmsf as rmsf
import numpy as np
import matplotlib.pyplot as plt
import os

In [ ]:
# Update these paths when you have CHARMM-GUI/OpenMM output
PSF = "../charmm-gui/step3_input.psf"
DCD_WT = "../simulations/wt/output.dcd"
DCD_MUT = "../simulations/i825t/output.dcd"

u_wt  = mda.Universe(PSF, DCD_WT)
u_mut = mda.Universe(PSF, DCD_MUT)
ref   = mda.Universe(PSF, DCD_WT)  # first frame as reference

In [ ]:
# Trigger loop: residues 1078-1089 chain A
tl_sel = "backbone and resid 1078:1089 and segid PROA"

R_wt = rms.RMSD(u_wt, ref, select=tl_sel)
R_wt.run()

R_mut = rms.RMSD(u_mut, ref, select=tl_sel)
R_mut.run()

plt.figure(figsize=(10,4))
plt.plot(R_wt.results.rmsd[:,1],  label="WT",   color="blue")
plt.plot(R_mut.results.rmsd[:,1], label="I825T", color="red")
plt.xlabel("Frame")
plt.ylabel("RMSD (Å)")
plt.title("Trigger Loop RMSD")
plt.legend()
plt.tight_layout()
plt.savefig("../analysis/TL_RMSD.png", dpi=150)
plt.show()

In [ ]:
bh_sel = "backbone and resid 824:844 and segid PROA"
tl_sel = "backbone and resid 1078:1089 and segid PROA"

for label, sel in [("TriggerLoop", tl_sel), ("BridgeHelix", bh_sel)]:
    for tag, u in [("WT", u_wt), ("I825T", u_mut)]:
        atoms = u.select_atoms(sel)
        R = rms.RMSF(atoms).run()
        plt.plot(R.results.rmsf, label=tag)
    plt.title(f"RMSF — {label}")
    plt.xlabel("Residue index")
    plt.ylabel("RMSF (Å)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"../analysis/{label}_RMSF.png", dpi=150)
    plt.show()

In [ ]:
# Key GTP contact distances from Dutagaci 2023
distance_pairs = {
    "GTP-R766":  ("resid 766 and name NZ  and segid PROA", "resname GTP and name PG"),
    "GTP-R1020": ("resid 1020 and name NH2 and segid PROA", "resname GTP and name O1G"),
    "GTP-H1085": ("resid 1085 and name ND1 and segid PROA", "resname GTP and name N1"),
    "GTP-L1081": ("resid 1081 and name CA  and segid PROA", "resname GTP and name C8"),
    "GTP-Mg":    ("resname MG and name MG",                  "resname GTP and name O1A"),
}

def calc_distances(u, pairs):
    results = {k: [] for k in pairs}
    for ts in u.trajectory:
        for name, (sel1, sel2) in pairs.items():
            a1 = u.select_atoms(sel1).positions
            a2 = u.select_atoms(sel2).positions
            if len(a1) > 0 and len(a2) > 0:
                dist = np.linalg.norm(a1[0] - a2[0])
                results[name].append(dist)
    return results

d_wt  = calc_distances(u_wt,  distance_pairs)
d_mut = calc_distances(u_mut, distance_pairs)

fig, axes = plt.subplots(len(distance_pairs), 1, figsize=(10, 3*len(distance_pairs)))
for ax, (name, vals_wt) in zip(axes, d_wt.items()):
    ax.plot(vals_wt,          label="WT",   color="blue")
    ax.plot(d_mut[name],      label="I825T", color="red")
    ax.set_title(f"Distance: {name}")
    ax.set_ylabel("Distance (Å)")
    ax.legend()
plt.tight_layout()
plt.savefig("../analysis/GTP_distances.png", dpi=150)
plt.show()

In [ ]:
latch_pairs = {
    "R728-D760": ("resid 728 and name NH2 and segid PROA", "resid 760 and name OD1 and segid PROA"),
    "R728-L808": ("resid 728 and name NH2 and segid PROA", "resid 808 and name O   and segid PROA"),
}

d_latch_wt  = calc_distances(u_wt,  latch_pairs)
d_latch_mut = calc_distances(u_mut, latch_pairs)

for name in latch_pairs:
    plt.figure(figsize=(10,3))
    plt.plot(d_latch_wt[name],  label="WT",   color="blue")
    plt.plot(d_latch_mut[name], label="I825T", color="red")
    plt.title(f"Latch Distance: {name}")
    plt.ylabel("Distance (Å)")
    plt.legend()
    plt.tight_layout()
    plt.savefig(f"../analysis/Latch_{name}.png", dpi=150)
    plt.show()
    